# SAGE — YOLOv3 Training on Google Colab

This notebook trains a YOLOv3 model on the WIT-UAS + HIT-UAV infrared thermal
drone datasets, then exports to ONNX for edge deployment on a Raspberry Pi.

**Before you start:** Go to **Runtime → Change runtime type → GPU (T4)**

## 1 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available — go to Runtime → Change runtime type → GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 2 — Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/castacks/WIT-UAS-Dataset.git
%cd WIT-UAS-Dataset

# Initialize HIT-UAV submodule (additional thermal dataset)
!git submodule update --init --recursive

In [ ]:
# Install required packages not already in Colab
!pip install -q minio albumentations terminaltables torchsummary wandb

## 3 — Download Datasets

In [ ]:
from minio import Minio
import os

endpoint_url = "airlab-share-02.andrew.cmu.edu:9000"
bucket_name = "wit-uas-dataset"

client = Minio(
    endpoint_url,
    access_key="y5zkhCOAl5d0OMxwtx4H",
    secret_key="W5qsLEyLYNofJhupZ0S4ofarEIAHDm9wkwV5UHFs",
    secure=True,
)

for f in ["WIT-UAV-Dataset.zip", "WIT-UAV-Dataset_split.zip"]:
    if not os.path.exists(f):
        print(f"Downloading {f}...")
        client.fget_object(bucket_name, f, f)
        print(f"✓ {f} downloaded")
    else:
        print(f"✓ {f} already exists")

In [ ]:
import zipfile

for f in ["WIT-UAV-Dataset.zip", "WIT-UAV-Dataset_split.zip"]:
    print(f"Extracting {f}...")
    with zipfile.ZipFile(f, "r") as z:
        z.extractall(".")
    print(f"✓ {f} extracted")

# Verify directory structure
assert os.path.isdir("WIT-UAV-Dataset_split/train"), "WIT train split missing"
assert os.path.isdir("WIT-UAV-Dataset_split/val"), "WIT val split missing"
print("\n✓ Dataset directories verified")

# Check HIT-UAV submodule
hit_dir = "HIT-UAV-Infrared-Thermal-Dataset/normal_json"
if os.path.isdir(hit_dir):
    print(f"✓ HIT-UAV dataset present at {hit_dir}")
else:
    print(f"⚠ HIT-UAV submodule empty — training with WIT data only")

## 4 — Inspect Dataset

In [ ]:
# Show classes
with open("dataset.classes", "r") as f:
    classes = f.read().strip().splitlines()
print("Classes:")
for i, c in enumerate(classes):
    print(f"  {i}: {c}")

# Count training images
wit_train = "WIT-UAV-Dataset_split/train"
sensors = [d for d in os.listdir(wit_train) if os.path.isdir(os.path.join(wit_train, d))]
total = 0
for s in sensors:
    imgs = [f for f in os.listdir(os.path.join(wit_train, s)) if f.endswith((".jpg", ".png", ".jpeg"))]
    print(f"  {s}: {len(imgs)} images")
    total += len(imgs)
print(f"\nTotal WIT training images: {total}")

## 5 — Mount Google Drive (Save Checkpoints)

Colab sessions disconnect after ~12 hrs. Mount Drive so checkpoints persist.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Create a folder on Drive for checkpoints
DRIVE_CKPT = "/content/drive/MyDrive/SAGE_checkpoints"
os.makedirs(DRIVE_CKPT, exist_ok=True)
print(f"✓ Checkpoints will be backed up to: {DRIVE_CKPT}")

## 6 — Train YOLOv3

Training uses the repo's own `train.py`. We call it from the command line.

**Key arguments:**
- `--data all` → trains on both HIT-UAV + WIT-UAV combined dataset
- `--data wit` → trains on WIT-UAV only (use this if HIT-UAV submodule is empty)
- `--epochs 300` → good balance of quality vs Colab time (~3-5 hrs on T4)
- `--batch-size 16` → fits in T4 16GB VRAM at 416×416
- `--checkpoint-interval 25` → saves every 25 epochs

In [ ]:
# Choose dataset based on whether HIT-UAV is available
DATA_FLAG = "all" if os.path.isdir("HIT-UAV-Infrared-Thermal-Dataset/normal_json/train") else "wit"
print(f"Training with: --data {DATA_FLAG}")

In [ ]:
!python -m model.yolo.train \
    --name sage-yolo-v1 \
    --model ./model/yolo/yolov3-custom.cfg \
    --data {DATA_FLAG} \
    --wit-sensor both \
    --epochs 300 \
    --batch-size 16 \
    --n-cpu 2 \
    --checkpoint-interval 25 \
    --evaluation-interval 10 \
    --seed 42

## 7 — Backup Checkpoints to Google Drive

In [ ]:
import glob, shutil

# Find the latest log directory (training saves checkpoints there)
log_dirs = sorted(glob.glob("logs/*"))
if log_dirs:
    latest_log = log_dirs[-1]
    ckpts = glob.glob(os.path.join(latest_log, "*.pth"))
    print(f"Found {len(ckpts)} checkpoints in {latest_log}")
    for c in ckpts:
        dest = os.path.join(DRIVE_CKPT, os.path.basename(c))
        shutil.copy2(c, dest)
        print(f"  ✓ Copied {os.path.basename(c)} to Drive")
else:
    # Also check local checkpoints/ dir
    ckpts = glob.glob("checkpoints/*.pth")
    print(f"Found {len(ckpts)} checkpoints in checkpoints/")
    for c in ckpts:
        dest = os.path.join(DRIVE_CKPT, os.path.basename(c))
        shutil.copy2(c, dest)
        print(f"  ✓ Copied {os.path.basename(c)} to Drive")

print(f"\n✓ All checkpoints backed up to {DRIVE_CKPT}")

## 8 — Evaluate Best Checkpoint

In [ ]:
# Find the latest (highest epoch) checkpoint
import re

all_ckpts = glob.glob("logs/**/*.pth", recursive=True) + glob.glob("checkpoints/*.pth")
if all_ckpts:
    # Sort by epoch number
    def extract_epoch(p):
        m = re.search(r"(\d+)\.pth$", p)
        return int(m.group(1)) if m else 0
    best_ckpt = max(all_ckpts, key=extract_epoch)
    print(f"Evaluating: {best_ckpt}")
else:
    raise FileNotFoundError("No checkpoints found — did training complete?")

In [ ]:
!python -m model.yolo.test \
    --model ./model/yolo/yolov3-custom.cfg \
    --weights {best_ckpt} \
    --batch_size 8 \
    --conf_thres 0.5 \
    --nms_thres 0.4

## 9 — Export to ONNX

Convert the trained PyTorch model to ONNX for edge deployment on
Raspberry Pi (runs with ONNX Runtime — no PyTorch needed).

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
import sys, os
sys.path.insert(0, ".")
sys.path.insert(0, "./model/yolo")

import torch
import numpy as np
from model.yolo.model import load_model

# Load trained model
model = load_model("./model/yolo/yolov3-custom.cfg", best_ckpt)
model.eval()
model.cpu()

# Dummy input (batch=1, 3 channels, 416x416)
dummy = torch.randn(1, 3, 416, 416)

ONNX_PATH = "sage-yolov3.onnx"

torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    opset_version=11,
    input_names=["images"],
    output_names=["detections"],
    dynamic_axes={
        "images": {0: "batch"},
        "detections": {0: "batch"},
    },
)

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f"✓ Exported {ONNX_PATH} ({size_mb:.1f} MB)")

In [ ]:
# Validate ONNX — run inference and compare to PyTorch
import onnxruntime as ort
import onnx

# Check ONNX model is valid
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("✓ ONNX model is valid")

# Run inference with ONNX Runtime
session = ort.InferenceSession(ONNX_PATH)
test_input = np.random.randn(1, 3, 416, 416).astype(np.float32)

ort_outputs = session.run(None, {"images": test_input})
print(f"✓ ONNX inference output shape: {ort_outputs[0].shape}")

# Compare with PyTorch
with torch.no_grad():
    pt_output = model(torch.from_numpy(test_input))
print(f"✓ PyTorch inference output shape: {pt_output.shape}")

# Check outputs are close
diff = np.abs(ort_outputs[0] - pt_output.numpy()).max()
print(f"Max absolute difference: {diff:.6f}")
assert diff < 1e-3, f"Outputs differ too much: {diff}"
print("✓ ONNX and PyTorch outputs match!")

## 10 — Save ONNX to Google Drive

In [ ]:
import shutil

drive_onnx = os.path.join(DRIVE_CKPT, "sage-yolov3.onnx")
shutil.copy2(ONNX_PATH, drive_onnx)
print(f"✓ ONNX model saved to Google Drive: {drive_onnx}")

# Also copy best checkpoint
drive_best = os.path.join(DRIVE_CKPT, "best_checkpoint.pth")
shutil.copy2(best_ckpt, drive_best)
print(f"✓ Best checkpoint saved to Google Drive: {drive_best}")

## 11 — Download Model Files

Download the ONNX model to your local machine, then copy it to your
Raspberry Pi's `inference/` folder.

In [ ]:
from google.colab import files

# Download ONNX model
files.download(ONNX_PATH)
print("\n✓ Download started — save sage-yolov3.onnx to sage-dashboard/inference/")

In [ ]:
# Also download the dataset.classes file (needed by inference script)
files.download("dataset.classes")

## 12 — Visualize Some Detections

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

def run_detection_demo(onnx_path, image_path, classes, conf_thres=0.5, nms_thres=0.4):
    """Run ONNX inference on a single image and display results."""
    session = ort.InferenceSession(onnx_path)

    # Load and preprocess image
    img_raw = cv2.imread(image_path)
    img_raw = cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img_raw.shape[:2]

    img = cv2.resize(img_raw, (416, 416))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))  # HWC → CHW
    img = np.expand_dims(img, 0)  # Add batch dim

    # Run inference
    outputs = session.run(None, {"images": img})
    detections = outputs[0]  # shape: [batch, num_detections, 5+num_classes]

    # Parse detections
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img_raw)

    if detections is not None and len(detections) > 0:
        det = detections[0]  # First batch item
        # Filter by objectness confidence
        obj_conf = det[:, 4]
        mask = obj_conf > conf_thres
        det = det[mask]

        if len(det) > 0:
            # Get class predictions
            cls_conf = det[:, 5:]
            cls_ids = np.argmax(cls_conf, axis=1)
            cls_scores = np.max(cls_conf, axis=1)

            colors = ['red', 'lime', 'blue', 'cyan', 'magenta', 'yellow']
            for i in range(len(det)):
                x_c, y_c, w, bh = det[i, :4]
                x1 = (x_c - w/2) * w_orig / 416
                y1 = (y_c - bh/2) * h_orig / 416
                bw = w * w_orig / 416
                bbh = bh * h_orig / 416
                cls_id = cls_ids[i]
                rect = patches.Rectangle((x1, y1), bw, bbh, linewidth=2,
                                          edgecolor=colors[cls_id % len(colors)],
                                          facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, f"{classes[cls_id]} {obj_conf[mask][i]:.2f}",
                        color='white', fontsize=9,
                        bbox=dict(boxstyle='round,pad=0.2', facecolor=colors[cls_id % len(colors)], alpha=0.7))

    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Find a sample image from the dataset
sample_dirs = glob.glob("WIT-UAV-Dataset_split/val/*/")
if sample_dirs:
    sample_images = glob.glob(os.path.join(sample_dirs[0], "*.jpg")) + \
                    glob.glob(os.path.join(sample_dirs[0], "*.png"))
    if sample_images:
        print(f"Running detection on: {sample_images[0]}")
        run_detection_demo(ONNX_PATH, sample_images[0], classes)
    else:
        print("No sample images found in val split")
else:
    print("No val directories found")

---

## Next Steps

1. Download `sage-yolov3.onnx` from Google Drive to your local machine
2. Copy it to `sage-dashboard/inference/sage-yolov3.onnx`
3. Copy `dataset.classes` to `sage-dashboard/inference/dataset.classes`
4. On the Raspberry Pi, run:
   ```bash
   cd sage-dashboard/inference
   pip install -r requirements.txt
   python detect_and_post.py --model sage-yolov3.onnx --server http://<your-pc-ip>:5000
   ```